# 판다스(Pandas) DAY 2 — 심화 과정

> **"깨끗한 데이터가 있어야 똑똑한 AI가 나온다."**

---

## 오늘의 학습 목표

| 순서 | 주제 | 담당 Ping / Pong |
|:---:|:---|:---|
| 1부 | **결측값 처리** · **타입 변환** · **날짜 다루기** | Ping 1~3 / Pong 1~3 |
| 2부 | **apply/map/lambda** · **문자열 메서드** · **데이터 결합** | Ping 4~6 / Pong 4~6 |
| 3부 | **groupby 심화** · **crosstab** · **pivot_table** · **stack/unstack/melt** | Ping 7~10 / Pong 7~10 |

---

## 오늘의 데이터셋

| 구분 | 이름 | 설명 | 규모 |
|:---:|:---|:---|:---|
| **Ping**(시범) | 심장질환 임상 데이터 `heart.csv` | 환자별 임상 지표와 심장질환 여부 | 918행 × 12열 |
| **Pong**(실습) | LoL 다이아몬드 랭크 `high_diamond_ranked_10min.csv` | 게임 10분 시점의 블루/레드 팀 지표와 승패 | 9,879행 × 40열 |

---

## DAY 2의 위치 — AI 파이프라인

```
원본 CSV
    ↓  pd.read_csv()   ← DAY 1에서 완료
 판다스 데이터프레임
    ↓  결측값 처리      ← DAY 2 1부
    ↓  타입 변환        ← DAY 2 1부
    ↓  피처 엔지니어링  ← DAY 2 2부 (apply/map)
    ↓  데이터 결합      ← DAY 2 2부 (merge)
    ↓  집계/재구조화    ← DAY 2 3부 (groupby/pivot)
 깨끗한 2차원 배열
    ↓  .values
 넘파이(NumPy)
    ↓  torch.tensor()
 파이토치 텐서
    ↓
 AI 모델 훈련
```

→ 실제 AI 프로젝트에서 DAY 2의 기술들이 **전체 데이터 준비의 70%**를 차지한다.

---
## 환경 설정

아래 셀을 먼저 실행하라.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

try:
    matplotlib.rcParams['font.family'] = 'NanumGothic'
except Exception:
    pass
matplotlib.rcParams['axes.unicode_minus'] = False

print("=" * 55)
print("  DAY 2 — 판다스 심화 과정")
print("  Ping : heart.csv (심장질환 임상 데이터)")
print("  Pong : high_diamond_ranked_10min.csv (LoL)")
print("=" * 55)
print(f"→ 판다스 버전 : {pd.__version__}")
print(f"→ 넘파이 버전 : {np.__version__}")
print("→ 환경 설정 완료")

---
## 실습 데이터 로드

GitHub에서 직접 불러온다. 접속 오류 시 합성 데이터로 자동 대체된다.

| 구분 | URL |
|:---:|:---|
| heart.csv (Ping) | `.../AI프로그래밍/data/heart.csv` |
| LoL (Pong) | `.../AI프로그래밍/data/high_diamond_ranked_10min.csv` |

In [ ]:
import random

BASE = "https://raw.githubusercontent.com/leina99-lab/classes/main/AI%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D/data/"

# ── Ping 데이터: heart.csv ─────────────────────────────────────
try:
    df_heart = pd.read_csv(BASE + "heart.csv")
    print(f"→ heart.csv 로드 성공: {df_heart.shape}")
except Exception:
    # 합성 데이터 fallback — 고유값 보장
    random.seed(42)
    n = 918
    ages      = [30 + (i*7 + 11) % 50 for i in range(n)]
    bps       = [100 + (i*3 + 7) % 80 for i in range(n)]
    chols     = [150 + (i*11 + 13) % 200 for i in range(n)]
    max_hrs   = [120 + (i*5 + 3) % 80 for i in range(n)]
    oldpeaks  = [round((i*0.17) % 4.0, 1) for i in range(n)]
    sexes     = ['M' if i%3 != 0 else 'F' for i in range(n)]
    cps       = ['ATA','NAP','ASY','TA'][ : ]
    chest     = [cps[i%4] for i in range(n)]
    disease   = [(i*3+1)%2 for i in range(n)]

    df_heart = pd.DataFrame({
        'Age': ages, 'Sex': sexes, 'ChestPainType': chest,
        'RestingBP': bps, 'Cholesterol': chols, 'MaxHR': max_hrs,
        'Oldpeak': oldpeaks, 'HeartDisease': disease
    })
    print(f"→ heart.csv 대체(합성) 데이터 생성: {df_heart.shape}")

# ── Pong 데이터: LoL 10분 데이터 ──────────────────────────────
try:
    df_lol = pd.read_csv(BASE + "high_diamond_ranked_10min.csv")
    print(f"→ LoL 데이터 로드 성공: {df_lol.shape}")
except Exception:
    random.seed(7)
    n = 9879
    wins       = [(i*7+3)%2 for i in range(n)]
    kills      = [(i*3+1)%15 for i in range(n)]
    deaths     = [(i*5+2)%15 for i in range(n)]
    assists    = [(i*4+1)%20 for i in range(n)]
    dragons    = [(i*11)%3 for i in range(n)]  # 0,1,2
    towers     = [(i*13+1)%4 for i in range(n)]
    gold_diff  = [(i*29+7) % 6000 - 3000 for i in range(n)]
    exp_diff   = [(i*31+3) % 6000 - 3000 for i in range(n)]
    first_blood= [(i*17+1)%2 for i in range(n)]
    df_lol = pd.DataFrame({
        'gameId': range(1, n+1),
        'blueWins': wins, 'blueFirstBlood': first_blood,
        'blueKills': kills, 'blueDeaths': deaths, 'blueAssists': assists,
        'blueDragons': dragons, 'blueTowersDestroyed': towers,
        'blueGoldDiff': gold_diff, 'blueExperienceDiff': exp_diff,
    })
    print(f"→ LoL 대체(합성) 데이터 생성: {df_lol.shape}")

print()
print("▶ heart.csv 첫 5행:")
print(df_heart.head())
print()
print("▶ LoL 데이터 첫 5행:")
print(df_lol.head())

---
# 1부: 데이터 정제 — 결측값 · 타입 · 날짜

> **핵심 질문**: *"지저분한 원본 데이터를 어떻게 AI가 먹을 수 있는 형태로 다듬는가?"*

---

## 1부 학습 맵

```
  원본 데이터 (heart.csv / LoL)
        ↓
  ① 결측값 확인 → 채우기 or 삭제     (Ping 1 / Pong 1)
        ↓
  ② 데이터 타입 점검 → 변환           (Ping 2 / Pong 2)
        ↓
  ③ 날짜 문자열 → 날짜 타입 → 연/월/일 (Ping 3 / Pong 3)
        ↓
  정제된 데이터 → 2부로 전달
```

## 1-1. 결측값 처리 — `isna()` · `fillna()` · `dropna()`

### 왜 중요한가?

**결측값**(missing value, `NaN`)은 비어있는 셀이다. 실제 데이터는 거의 **반드시** 결측값을 포함한다.

| 원인 | 예 |
|:---|:---|
| 측정 누락 | 환자가 혈압 측정을 거부함 |
| 센서 오류 | 스마트워치 배터리가 방전됨 |
| 통합 불일치 | 두 파일을 합칠 때 한쪽에만 있는 열 |

**판다스가 결측값을 처리하는 3대 전략**이다.

| 전략 | 함수 | 언제 쓰는가 |
|:---|:---|:---|
| **삭제** | `df.dropna()` | 결측이 매우 적거나(<5%) 행/열 전체가 비었을 때 |
| **채우기** | `df.fillna(값)` | 결측이 의미 있고, 평균·최빈값·중앙값 등으로 대체 가능할 때 |
| **표시** | `df.isna()` / `df.notna()` | 결측 여부를 불리언 마스크로 얻고 싶을 때 |

> **AI 연결**: 딥러닝 모델은 `NaN`이 한 칸만 있어도 **전체 출력이 `NaN`**이 된다. 결측값 처리는 AI 훈련 직전 필수 관문이다.

### Ping 1 — 결측값 처리 (heart.csv)

In [ ]:
# ─── Ping 1: 결측값 탐색과 처리 ──────────────────────────────
df1 = df_heart.copy()

# 일부러 결측값을 주입 (학습용)
np.random.seed(0)
mask = np.random.rand(len(df1)) < 0.05       # 약 5% 행
df1.loc[mask, 'Cholesterol'] = np.nan
mask2 = np.random.rand(len(df1)) < 0.03      # 약 3% 행
df1.loc[mask2, 'RestingBP'] = np.nan

print("▶ 결측값 수 (열별)")
print(df1.isna().sum())
print()
print(f"▶ 결측값 비율 (Cholesterol): {df1['Cholesterol'].isna().mean()*100:.2f}%")
print()

# [전략 1] 삭제 — dropna
df1_drop = df1.dropna()
print(f"▶ dropna()   후 행 수: {len(df1_drop):,}  (원본 {len(df1):,}에서 {len(df1)-len(df1_drop)}행 삭제)")

# [전략 2] 채우기 — 평균으로
chol_mean = df1['Cholesterol'].mean()
df1_fill = df1.fillna({'Cholesterol': chol_mean, 'RestingBP': df1['RestingBP'].median()})
print(f"▶ fillna()   후 결측 수: {df1_fill.isna().sum().sum()}")
print(f"▶ Cholesterol 평균: {chol_mean:.2f}로 채움")

# [전략 3] 마스크로 행 추출 — isna
missing_rows = df1[df1['Cholesterol'].isna()]
print(f"▶ isna()로 결측 행만 추출: {len(missing_rows)}행")

# [AI 연결]
print()
print("[AI 연결] 사이킷런(sklearn)의 SimpleImputer가 fillna의 일반화된 버전이다.")
print("         전략: 'mean', 'median', 'most_frequent', 'constant' 중 선택.")

### ✎ 개념 확인 1

**질문**: `df.isna().sum()`의 반환 타입은 무엇인가?

<details>
<summary>▶ 정답 보기</summary>

`pandas.Series`이다. 각 열 이름이 인덱스, 결측값 개수가 값인 시리즈가 반환된다. 따라서 바로 `.sort_values()`나 시각화(`.plot.bar()`)를 붙일 수 있다.
</details>

### Pong 1 — 결측값 처리 (LoL 데이터)

**과제**: LoL 데이터프레임에 결측값을 일부러 넣고, 세 가지 전략으로 각각 처리해보라.

1. `blueKills` 열의 10%를 `NaN`으로 만든 새 데이터프레임 `df_l1`을 만든다. (seed=42)
2. `df_l1.isna().sum()`을 출력하여 결측 수를 확인한다.
3. `dropna()`로 삭제한 결과의 행 수를 출력한다.
4. `fillna()`로 `blueKills`의 **중앙값**을 채운 결과를 확인한다.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l1 = df_lol.copy()
np.random.seed(42)
mask = np.random.rand(len(df_l1)) < 0.10
df_l1.loc[mask, 'blueKills'] = np.nan

print("▶ 결측값 수:")
print(df_l1.isna().sum())
print()
print(f"▶ dropna() 후 행 수: {len(df_l1.dropna()):,}")
print()
median_kills = df_l1['blueKills'].median()
df_l1_fill = df_l1.fillna({'blueKills': median_kills})
print(f"▶ 중앙값 {median_kills}로 채움 → 결측 수: {df_l1_fill.isna().sum().sum()}")
```
</details>

In [ ]:
# ─── Pong 1: 여기에 코드를 작성하시오 ────────────────────────


---
## 1-2. 타입 변환 — `astype()` · `to_numeric()` · `to_datetime()`

### 왜 중요한가?

판다스는 열마다 **데이터 타입**(dtype)을 가진다. 타입이 맞지 않으면 연산이 실패하거나 조용히 잘못된 결과를 낸다.

| 타입 | 설명 | 변환 함수 |
|:---|:---|:---|
| `int64` / `int32` | 정수 | `astype('int')` |
| `float64` | 실수 | `astype('float')` |
| `object` | 문자열(혹은 혼합) | `astype('str')` |
| `bool` | 참/거짓 | `astype('bool')` |
| `category` | 범주형(메모리 절약) | `astype('category')` |
| `datetime64[ns]` | 날짜/시각 | `pd.to_datetime(...)` |

> **실무 함정**: 엑셀에서 읽은 `"1,234"` 같은 값은 쉼표 때문에 `object`로 읽힌다. `astype(int)`는 실패한다. 쉼표를 먼저 제거하고 `pd.to_numeric(errors='coerce')`를 쓰는 것이 안전하다.

> **AI 연결**: 딥러닝 입력은 **float32**가 표준이다. `df.astype('float32')`로 바꿔야 GPU 메모리를 절약할 수 있다.

### Ping 2 — 타입 변환 (heart.csv)

In [ ]:
# ─── Ping 2: 타입 변환 ───────────────────────────────────────
df2 = df_heart.copy()
print("▶ 원본 타입")
print(df2.dtypes)
print()

# [1] int → float (딥러닝 입력용)
df2['Age_float'] = df2['Age'].astype('float32')
print(f"▶ Age_float dtype: {df2['Age_float'].dtype}")

# [2] object → category (메모리 절약)
df2['Sex_cat'] = df2['Sex'].astype('category')
print(f"▶ Sex_cat dtype    : {df2['Sex_cat'].dtype}")
print(f"▶ Sex_cat 카테고리 : {df2['Sex_cat'].cat.categories.tolist()}")

# 메모리 비교
mem_before = df2['Sex'].memory_usage(deep=True)
mem_after  = df2['Sex_cat'].memory_usage(deep=True)
print(f"▶ 메모리: object={mem_before:,}B  →  category={mem_after:,}B")
print(f"  절감률: {(1-mem_after/mem_before)*100:.1f}%")
print()

# [3] bool 변환 (조건의 결과를 정식 타입으로)
df2['HasDisease'] = df2['HeartDisease'].astype('bool')
print(f"▶ HasDisease dtype: {df2['HasDisease'].dtype}")
print(df2[['HeartDisease','HasDisease']].head())

# [4] to_numeric — 안전하게 수치 변환 (errors='coerce')
messy = pd.Series(['120','130','N/A','145','unknown','110'])
clean = pd.to_numeric(messy, errors='coerce')
print()
print("▶ 더러운 문자 시리즈 → 수치 변환")
print(f"  원본: {messy.tolist()}")
print(f"  변환: {clean.tolist()}  (실패는 NaN)")

# [AI 연결]
print()
print("[AI 연결] 파이토치의 torch.float32가 표준이다.")
print("         df.astype('float32').values → torch.tensor()")

### ✎ 개념 확인 2

**질문**: `pd.to_numeric(series, errors='coerce')`에서 `errors='coerce'`의 역할은?

<details>
<summary>▶ 정답 보기</summary>

변환 실패 값을 `NaN`으로 대체한다. 반대로 `errors='raise'`(기본)은 오류 발생 시 예외를 던지고, `errors='ignore'`는 원본을 그대로 반환한다. 실무에서는 대체로 `'coerce'`를 쓴 뒤 `fillna`로 후처리한다.
</details>

### Pong 2 — 타입 변환 (LoL 데이터)

**과제**:
1. `df_lol`을 복사하여 `df_l2`를 만든다.
2. `blueGoldDiff`를 `float32`로 변환하여 `blueGoldDiff_f` 열로 저장한다.
3. `blueFirstBlood`를 **bool 타입**으로 변환하여 `FirstBlood_bool` 열로 저장한다.
4. 두 변환 전후의 dtype을 모두 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l2 = df_lol.copy()
df_l2['blueGoldDiff_f']   = df_l2['blueGoldDiff'].astype('float32')
df_l2['FirstBlood_bool']  = df_l2['blueFirstBlood'].astype('bool')

print(f"blueGoldDiff       : {df_l2['blueGoldDiff'].dtype}")
print(f"blueGoldDiff_f     : {df_l2['blueGoldDiff_f'].dtype}")
print(f"blueFirstBlood     : {df_l2['blueFirstBlood'].dtype}")
print(f"FirstBlood_bool    : {df_l2['FirstBlood_bool'].dtype}")
print(df_l2[['blueFirstBlood','FirstBlood_bool']].head())
```
</details>

In [ ]:
# ─── Pong 2: 여기에 코드를 작성하시오 ────────────────────────


---
## 1-3. 날짜 다루기 — `pd.to_datetime()` · `.dt` 접근자

### 왜 중요한가?

CSV에서 날짜는 대개 `"2024-03-15"` 같은 **문자열**로 들어온다. 이 상태로는 월별 집계, 요일별 집계, 기간 계산이 불가능하다. **반드시 `datetime64` 타입으로 변환**해야 시계열 분석이 시작된다.

### 날짜 변환의 3단계

```
  "2024-03-15"  (문자열)
        ↓  pd.to_datetime()
  Timestamp('2024-03-15')  (datetime64)
        ↓  .dt.year / .dt.month / .dt.dayofweek
  2024, 3, 4 (= 목요일)   (수치/범주)
```

### `.dt` 접근자로 꺼낼 수 있는 것

| 속성 | 반환 | 예 (`2024-03-15`) |
|:---|:---|:---|
| `.dt.year` | 연도 | 2024 |
| `.dt.month` | 월 (1~12) | 3 |
| `.dt.day` | 일 (1~31) | 15 |
| `.dt.dayofweek` | 요일 (0=월 ~ 6=일) | 4 |
| `.dt.day_name()` | 요일 이름 | `'Friday'` |
| `.dt.quarter` | 분기 (1~4) | 1 |
| `.dt.hour` | 시 (0~23) | 0 |

> **AI 연결**: 시계열 예측 모델(LSTM, Transformer)에서 "월", "요일" 같은 주기 정보는 **별도 특성**으로 분리해 넣는 것이 표준이다.

### Ping 3 — 날짜 다루기 (가상 진료일 생성)

In [ ]:
# ─── Ping 3: 날짜 변환과 분해 ────────────────────────────────
df3 = df_heart.copy().head(10).reset_index(drop=True)

# 가상의 진료일을 문자열로 부여
visit_dates = ['2024-01-15','2024-02-03','2024-02-28','2024-03-11','2024-03-22',
               '2024-04-05','2024-04-18','2024-05-07','2024-05-23','2024-06-10']
df3['VisitDate'] = visit_dates
print("▶ 변환 전 dtype:", df3['VisitDate'].dtype)
print(df3[['Age','VisitDate']].head(3))
print()

# [핵심] 문자열 → datetime64
df3['VisitDate'] = pd.to_datetime(df3['VisitDate'])
print("▶ 변환 후 dtype:", df3['VisitDate'].dtype)
print()

# .dt 접근자로 분해
df3['Year']       = df3['VisitDate'].dt.year
df3['Month']      = df3['VisitDate'].dt.month
df3['Day']        = df3['VisitDate'].dt.day
df3['DayOfWeek']  = df3['VisitDate'].dt.dayofweek    # 0=월 ... 6=일
df3['DayName']    = df3['VisitDate'].dt.day_name()
df3['Quarter']    = df3['VisitDate'].dt.quarter

print("▶ 날짜 분해 결과:")
print(df3[['VisitDate','Year','Month','DayOfWeek','DayName','Quarter']].head())
print()

# 기간 계산 (오늘까지 며칠 지났는가)
today = pd.Timestamp.now().normalize()
df3['DaysSinceVisit'] = (today - df3['VisitDate']).dt.days
print(f"▶ 오늘: {today.date()}")
print(df3[['VisitDate','DaysSinceVisit']].head())
print()

# 월별 방문 수 집계
by_month = df3.groupby(df3['VisitDate'].dt.month).size()
print("▶ 월별 방문 수:")
print(by_month)

# [AI 연결]
print()
print("[AI 연결] 시계열 예측에서 월·요일·시간대를 별도 특성으로 분리하는 것이")
print("         Feature Engineering의 기본이다.")

### ✎ 개념 확인 3

**질문**: `.dt.dayofweek`에서 월요일은 어떤 숫자인가?

<details>
<summary>▶ 정답 보기</summary>

**0**이다. `0=월, 1=화, 2=수, 3=목, 4=금, 5=토, 6=일` 순이다. 반면 `.dt.weekday`도 동일한 규칙이다. `.dt.day_name()`은 영문 이름(`'Monday'` ...)을 반환한다.
</details>

### Pong 3 — 날짜 다루기 (LoL 가상 경기일)

**과제**:
1. `df_lol.head(10)`을 복사하여 `df_l3`를 만든다.
2. `['2024-11-01','2024-11-05','2024-11-12','2024-11-18','2024-11-25','2024-12-02','2024-12-09','2024-12-15','2024-12-22','2024-12-30']`을 `MatchDate` 열로 붙이고 datetime으로 변환한다.
3. 요일(`DayOfWeek`), 요일명(`DayName`), 월(`Month`)을 새 열로 생성한다.
4. 요일별 경기 수를 `value_counts()`로 집계한다.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l3 = df_lol.head(10).copy().reset_index(drop=True)
match_dates = ['2024-11-01','2024-11-05','2024-11-12','2024-11-18','2024-11-25',
               '2024-12-02','2024-12-09','2024-12-15','2024-12-22','2024-12-30']
df_l3['MatchDate']  = pd.to_datetime(match_dates)
df_l3['DayOfWeek']  = df_l3['MatchDate'].dt.dayofweek
df_l3['DayName']    = df_l3['MatchDate'].dt.day_name()
df_l3['Month']      = df_l3['MatchDate'].dt.month

print(df_l3[['MatchDate','DayOfWeek','DayName','Month']])
print()
print("▶ 요일별 경기 수:")
print(df_l3['DayName'].value_counts())
```
</details>

In [ ]:
# ─── Pong 3: 여기에 코드를 작성하시오 ────────────────────────


---
## 1부 연습문제 (10문항)

> 세션 외 숙제. 토글을 먼저 가려놓고 스스로 풀어보라.

In [ ]:
# ─── 연습문제 기본 데이터 ───────────────────────────────────
# 학생용 샘플 데이터를 생성한다.
practice = pd.DataFrame({
    'patient_id'  : ['P001','P002','P003','P004','P005','P006','P007','P008'],
    'age'         : [45, 52, np.nan, 60, 35, 58, np.nan, 41],
    'chol'        : [210, 240, 190, np.nan, 180, 260, 220, 195],
    'visit_date'  : ['2024-01-10','2024-02-15','2024-03-20','2024-04-05',
                     '2024-05-12','2024-06-18','2024-07-22','2024-08-30'],
    'sex'         : ['M','F','M','F','M','M','F','F'],
    'heart_disease': [1, 0, 1, 1, 0, 1, 0, 0]
})
print(practice)

### 연습문제 1. 결측 개수

`practice`에서 **열별 결측값 개수**를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice.isna().sum()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 2. 결측 제거

결측이 하나라도 있는 행을 모두 제거한 결과의 **행 수**를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
len(practice.dropna())
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 3. 평균으로 채우기

`age` 결측을 **평균**으로 채우고 결과 시리즈를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice['age'].fillna(practice['age'].mean())
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 4. 중앙값으로 채우기

`chol` 결측을 **중앙값**으로 채운 시리즈를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice['chol'].fillna(practice['chol'].median())
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 5. 타입 변환 — float32

`age`를 결측 제거 후 **float32**로 변환하고 dtype을 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice['age'].dropna().astype('float32').dtype
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 6. 타입 변환 — bool

`heart_disease`를 **bool**로 변환한 시리즈를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice['heart_disease'].astype('bool')
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 7. 문자열 → datetime

`visit_date`를 datetime으로 변환하고 dtype을 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.to_datetime(practice['visit_date']).dtype
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 8. 월 추출

`visit_date`에서 **월(month)**만 추출한 시리즈를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.to_datetime(practice['visit_date']).dt.month
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 9. 요일 이름

`visit_date`의 **요일명**을 영문으로 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.to_datetime(practice['visit_date']).dt.day_name()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 10. 범주형 변환

`sex`를 **category** 타입으로 바꾸고 카테고리 목록을 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
practice['sex'].astype('category').cat.categories.tolist()
```
</details>

In [ ]:
# 여기에 코드 작성


---
# 2부: 변환과 결합 — `apply` · 문자열 · `merge`

> **핵심 질문**: *"기존 열을 변형하여 새로운 특성을 만들고, 여러 파일을 하나로 합친다."*

---

## 2부 학습 맵

```
  ① apply / map / lambda — 사용자 정의 변환   (Ping 4 / Pong 4)
        예: 콜레스테롤 → "정상/경계/위험" 범주
        ↓  (AI 연결: 사이킷런의 FunctionTransformer와 동일 원리)

  ② 문자열 메서드 — 텍스트 열을 열어서 본다    (Ping 5 / Pong 5)
        예: 환자 메모에서 특정 키워드 추출
        ↓  (AI 연결: NLP 전처리의 첫 단계)

  ③ merge / concat / join — 파일을 합친다       (Ping 6 / Pong 6)
        예: 환자 정보 + 검사 결과를 patient_id로 결합
        ↓  (AI 연결: 피처 테이블을 여러 원천에서 모을 때)
```

## 2-1. `apply` · `map` · `lambda` — 사용자 정의 변환

### 세 함수의 역할 구분

| 함수 | 대상 | 전달하는 것 | 쓰임새 |
|:---|:---|:---|:---|
| `Series.map()` | **시리즈** | **딕셔너리 / 함수** | 범주형 코딩, 라벨 인코딩 |
| `Series.apply()` | **시리즈** | **함수** | 복잡한 변환 |
| `DataFrame.apply()` | **데이터프레임** | **함수** | 여러 열을 동시에 다루는 변환 |

### `lambda` 표현식

`lambda x: 식`은 **이름 없는 일회용 함수**이다.

```python
square = lambda x: x ** 2      #  def square(x): return x**2  와 같다
```

### AI 연결 — 이것은 곧 **ReLU**이다

```python
relu = lambda x: max(0, x)      # 딥러닝 활성화 함수 ReLU
df['z_relu'] = df['z'].apply(relu)
```

즉, `apply`로 시리즈 전체에 함수를 일괄 적용하는 것이 **신경망의 활성화 함수와 정확히 같은 연산**이다. 판다스와 파이토치는 같은 철학을 공유한다.

### Ping 4 — apply / map / lambda (heart.csv)

In [ ]:
# ─── Ping 4: apply / map / lambda ────────────────────────────
df4 = df_heart.copy()

# [1] map() + 딕셔너리 : 범주 → 숫자 (라벨 인코딩)
sex_map = {'M': 1, 'F': 0}
df4['Sex_code'] = df4['Sex'].map(sex_map)
print("▶ map()으로 라벨 인코딩")
print(df4[['Sex','Sex_code']].head())
print()

# [2] apply() + 함수 : 콜레스테롤 → 등급
def chol_grade(x):
    if x < 200:   return '정상'
    elif x < 240: return '경계'
    else:         return '위험'

df4['Chol_Grade'] = df4['Cholesterol'].apply(chol_grade)
print("▶ apply()로 등급 부여")
print(df4[['Cholesterol','Chol_Grade']].head(8))
print()
print("▶ 등급 분포:")
print(df4['Chol_Grade'].value_counts())
print()

# [3] apply() + lambda : 한 줄 변환
df4['Age_Group'] = df4['Age'].apply(lambda x: '40대이하' if x < 40 else ('60대이하' if x < 60 else '60대이상'))
print("▶ lambda로 나이 구간화")
print(df4['Age_Group'].value_counts())
print()

# [4] DataFrame.apply() : 여러 열을 동시에 — axis=1 필수
def risk_score(row):
    """환자 한 명의 위험 점수를 계산"""
    score = 0
    if row['Age'] > 55:         score += 1
    if row['Cholesterol'] > 240: score += 1
    if row['RestingBP'] > 140:   score += 1
    if row['MaxHR'] < 140:       score += 1
    return score

df4['Risk_Score'] = df4.apply(risk_score, axis=1)
print("▶ DataFrame.apply()로 여러 열을 합산")
print(df4[['Age','Cholesterol','RestingBP','MaxHR','Risk_Score']].head())
print()
print("▶ 위험 점수 분포:")
print(df4['Risk_Score'].value_counts().sort_index())

# [AI 연결]
print()
print("[AI 연결] lambda x: max(0, x) 는 딥러닝의 ReLU 활성화 함수다.")
print("         df['z'].apply(lambda x: max(0,x)) ≡ torch.relu(z) 와 같은 연산이다.")

### ✎ 개념 확인 4

**질문**: `DataFrame.apply(func)`에서 `axis=1`과 `axis=0`의 차이는?

<details>
<summary>▶ 정답 보기</summary>

- `axis=0` (기본): **각 열**을 시리즈 형태로 함수에 전달. 열 단위 집계에 쓴다.
- `axis=1`: **각 행**을 시리즈 형태로 함수에 전달. 여러 열을 조합해 새 값을 만들 때 쓴다.

즉, "열 방향으로 함수를 적용한다"라고 읽으면 `axis=0`, "행 방향으로 함수를 적용한다"라고 읽으면 `axis=1`이다. 헷갈리면 `axis=1 → 행 단위`로 외우면 된다.
</details>

### Pong 4 — apply / map / lambda (LoL 데이터)

**과제**:
1. `df_lol`을 복사해 `df_l4`를 만든다.
2. `blueGoldDiff`를 `apply`와 **lambda**로 `'Ahead'` / `'Even'` / `'Behind'`로 분류하여 `GoldStatus` 열에 저장한다. 기준: `>500 → Ahead`, `<-500 → Behind`, 그 외 `Even`.
3. `blueDragons`를 `map({0:'No Dragon', 1:'One', 2:'Two+'})`로 변환하여 `DragonLabel` 열에 저장한다.
4. **DataFrame.apply**로 `(blueKills + blueAssists - blueDeaths)`를 계산하여 `KDA_Score`를 만든다.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l4 = df_lol.copy()

df_l4['GoldStatus'] = df_l4['blueGoldDiff'].apply(
    lambda x: 'Ahead' if x > 500 else ('Behind' if x < -500 else 'Even')
)

df_l4['DragonLabel'] = df_l4['blueDragons'].map({0:'No Dragon', 1:'One', 2:'Two+'})

df_l4['KDA_Score'] = df_l4.apply(
    lambda r: r['blueKills'] + r['blueAssists'] - r['blueDeaths'], axis=1
)

print(df_l4[['blueGoldDiff','GoldStatus']].head())
print()
print(df_l4[['blueDragons','DragonLabel']].head())
print()
print(df_l4[['blueKills','blueAssists','blueDeaths','KDA_Score']].head())
```
</details>

In [ ]:
# ─── Pong 4: 여기에 코드를 작성하시오 ────────────────────────


---
## 2-2. 문자열 메서드 — `str` 접근자

### 왜 필요한가?

문자열 열은 단순 비교(`==`)만으로는 부족하다. **포함, 대소문자, 분리, 치환** 등 다양한 조작이 필요하다. 판다스는 파이썬의 문자열 메서드를 **시리즈 단위로 일괄 적용**하도록 `str` 접근자를 제공한다.

### 자주 쓰는 `str` 메서드

| 메서드 | 기능 | 예 |
|:---|:---|:---|
| `str.lower()` / `str.upper()` | 소문자 / 대문자 | `'Hello'` → `'hello'` |
| `str.contains(pat)` | 포함 여부 (불리언) | `'abc'.contains('b')` → `True` |
| `str.startswith(pat)` | 시작 여부 | |
| `str.endswith(pat)` | 끝남 여부 | |
| `str.replace(a, b)` | 치환 | `'abc'.replace('b','X')` → `'aXc'` |
| `str.split(sep)` | 분리 | `'a,b,c'.split(',')` → `['a','b','c']` |
| `str.len()` | 길이 | |
| `str.strip()` | 공백 제거 | |
| `str.extract(r'정규식')` | 정규식 추출 | |

> **AI 연결**: 자연어 처리(**NLP**)의 토큰화·정규화 단계가 이 메서드들로 시작한다. 챗봇 학습 데이터도 대부분 이렇게 정제된다.

### Ping 5 — 문자열 메서드 (heart.csv)

In [ ]:
# ─── Ping 5: 문자열 메서드 ──────────────────────────────────
df5 = df_heart.copy()

# 가상의 메모 열 추가
memos = [
    'chest pain after exercise',
    'mild headache, no chest pain',
    'severe Chest Pain at rest',
    'shortness of breath',
    'Chest pain radiating to left arm',
] * ((len(df5) // 5) + 1)
df5['Memo'] = memos[:len(df5)]

# [1] 대소문자 통일
df5['Memo_lower'] = df5['Memo'].str.lower()
print("▶ 소문자 통일")
print(df5[['Memo','Memo_lower']].head(3))
print()

# [2] 특정 단어 포함 여부
df5['HasChestPain'] = df5['Memo_lower'].str.contains('chest pain')
print(f"▶ 'chest pain' 포함 비율: {df5['HasChestPain'].mean()*100:.1f}%")
print(df5[['Memo','HasChestPain']].head(5))
print()

# [3] 길이
df5['Memo_len'] = df5['Memo'].str.len()
print(f"▶ 메모 평균 길이: {df5['Memo_len'].mean():.1f}자")
print()

# [4] 치환
df5['Memo_clean'] = df5['Memo_lower'].str.replace('chest', 'CHEST')
print("▶ 'chest' → 'CHEST' 치환")
print(df5[['Memo_lower','Memo_clean']].head(3))
print()

# [5] split + 첫 단어만 추출
df5['FirstWord'] = df5['Memo_lower'].str.split().str[0]
print("▶ 첫 단어 추출")
print(df5[['Memo_lower','FirstWord']].head(5))
print()

# [6] 범주 코드 추출 (ChestPainType의 첫 글자)
df5['CP_initial'] = df5['ChestPainType'].str[0]
print("▶ ChestPainType 첫 글자")
print(df5[['ChestPainType','CP_initial']].head())

# [AI 연결]
print()
print("[AI 연결] 챗봇 학습 데이터는 str.lower() → str.strip() → str.replace() 순으로")
print("         전처리된 뒤 토크나이저에 들어간다. 판다스가 NLP 파이프라인의 0단계다.")

### ✎ 개념 확인 5

**질문**: `df['col'].str.split(',')`의 반환 원소는 무엇인가?

<details>
<summary>▶ 정답 보기</summary>

**리스트**(list)이다. 각 셀마다 분리된 조각들이 파이썬 리스트로 담긴다. 그 뒤 `.str[0]`을 붙이면 리스트의 0번째 원소만 꺼내 시리즈로 만들 수 있다. 이를 **이중 `str` 접근**이라 한다.
</details>

### Pong 5 — 문자열 메서드 (LoL 데이터)

**과제**:
1. `df_l5 = df_lol.head(20).copy()`를 만든다.
2. `blueWins`의 0/1을 `'Win'`/`'Lose'`로 바꾸어 `Result` 열에 저장한다.
3. 새 메모 `['first blood won','dragon taken','herald killed','tower destroyed','gold ahead'] * 4`를 `Note`로 붙이고, `'dragon'`을 포함하는 행만 필터링해 출력한다.
4. `Note`의 첫 단어를 `FirstWord` 열로 추출한다.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l5 = df_lol.head(20).copy().reset_index(drop=True)

df_l5['Result'] = df_l5['blueWins'].map({0:'Lose', 1:'Win'})

notes = ['first blood won','dragon taken','herald killed','tower destroyed','gold ahead'] * 4
df_l5['Note'] = notes[:len(df_l5)]

has_dragon = df_l5[df_l5['Note'].str.contains('dragon')]
print("▶ 'dragon' 포함 행:")
print(has_dragon[['Result','Note']])

df_l5['FirstWord'] = df_l5['Note'].str.split().str[0]
print()
print("▶ 첫 단어 추출:")
print(df_l5[['Note','FirstWord']].head())
```
</details>

In [ ]:
# ─── Pong 5: 여기에 코드를 작성하시오 ────────────────────────


---
## 2-3. 데이터 결합 — `concat` · `merge` · `join`

### 세 가지 결합 방식

```
  concat (수직 이어 붙이기)       merge (공통 키로 가로 결합)
  ───────────────                ───────────────
     df_A                           df_환자      df_검사
     ────                           ──────      ──────
     1  x  y                        id │ age    id │ score
     2  a  b                        ───┼────    ───┼──────
                                    1  │ 30     1  │ 85
     +                              2  │ 45     2  │ 90
                                    3  │ 50     4  │ 70
     df_B
     ────                                  ↓ merge(on='id')
     3  x  y
     4  a  b                           id │ age │ score
                                       ───┼─────┼──────
     =                                  1 │ 30  │ 85
                                        2 │ 45  │ 90
     통합된 df
     ────
     1, 2, 3, 4
```

### `merge()`의 네 가지 join 방식

| `how=` | 의미 | 결과 행 수 |
|:---:|:---|:---|
| `'inner'` (기본) | **교집합** (양쪽에 모두 있는 키) | 가장 적음 |
| `'left'` | 왼쪽 기준 (오른쪽에 없으면 NaN) | 왼쪽 행 수 |
| `'right'` | 오른쪽 기준 | 오른쪽 행 수 |
| `'outer'` | **합집합** (어느 한쪽에만 있어도 포함) | 가장 많음 |

> **AI 연결**: 실제 ML 프로젝트는 대부분 **여러 원천 데이터를 merge**해서 피처 테이블을 만드는 것으로 시작한다.

### Ping 6 — merge / concat (heart.csv)

In [ ]:
# ─── Ping 6: merge / concat / join ──────────────────────────
# 환자 기본 정보
df_info = pd.DataFrame({
    'patient_id': ['P001','P002','P003','P004','P005'],
    'Age'       : [45, 52, 38, 60, 41],
    'Sex'       : ['M','F','M','F','M'],
})
print("▶ df_info (환자 정보)")
print(df_info)
print()

# 검사 결과 (P003, P006 불일치)
df_lab = pd.DataFrame({
    'patient_id' : ['P001','P002','P004','P005','P006'],
    'Cholesterol': [210, 240, 190, 260, 220],
    'RestingBP'  : [130, 145, 120, 155, 135],
})
print("▶ df_lab (검사 결과)")
print(df_lab)
print()

# [1] inner merge (교집합)
inner = pd.merge(df_info, df_lab, on='patient_id', how='inner')
print(f"▶ inner merge: {len(inner)}행 (공통 patient_id만)")
print(inner)
print()

# [2] left merge (df_info 기준)
left = pd.merge(df_info, df_lab, on='patient_id', how='left')
print(f"▶ left merge: {len(left)}행 (df_info 기준, 검사 없으면 NaN)")
print(left)
print()

# [3] outer merge (합집합)
outer = pd.merge(df_info, df_lab, on='patient_id', how='outer')
print(f"▶ outer merge: {len(outer)}행 (어느 한쪽에라도 있으면 포함)")
print(outer)
print()

# [4] concat (수직 — 행 추가)
more_info = pd.DataFrame({
    'patient_id': ['P007','P008'],
    'Age'       : [33, 55],
    'Sex'       : ['F','M'],
})
stacked = pd.concat([df_info, more_info], ignore_index=True)
print(f"▶ concat (수직): {len(df_info)} + {len(more_info)} = {len(stacked)}행")
print(stacked)

# [AI 연결]
print()
print("[AI 연결] 학습용 테이블은 보통 여러 원천을 merge한 결과다.")
print("         예: 사용자 테이블 + 행동 로그 + 상품 카탈로그 → 추천 모델 입력")

### ✎ 개념 확인 6

**질문**: `how='left'`와 `how='inner'`에서 결과 행 수가 다를 수 있는 이유는?

<details>
<summary>▶ 정답 보기</summary>

`left`는 **왼쪽 데이터프레임의 모든 행을 유지**하고, 오른쪽에 대응되는 키가 없으면 NaN을 채운다. `inner`는 **양쪽 모두에 있는 키**만 남긴다. 왼쪽에만 있는 키가 하나라도 있으면 두 결과의 행 수가 달라진다.
</details>

### Pong 6 — 데이터 결합 (LoL 데이터)

**과제**:
1. `df_lol.head(5)[['gameId','blueWins']].copy()`를 `df_a`로 만든다.
2. 다음과 같이 `df_b`를 만든다 — `gameId`는 1,2,3,6,7의 5개, 각 게임의 MVP 이름 `['A-카이','B-나미','C-진','D-리','E-마'] `.
3. `gameId`를 키로 `inner`/`left`/`outer` 세 방식으로 merge한 결과의 **행 수**를 비교한다.

<details>
<summary>▶ 정답 보기</summary>

```python
df_a = df_lol.head(5)[['gameId','blueWins']].copy()

df_b = pd.DataFrame({
    'gameId': [1,2,3,6,7],
    'MVP'   : ['A-카이','B-나미','C-진','D-리','E-마']
})

for how in ['inner','left','outer']:
    m = pd.merge(df_a, df_b, on='gameId', how=how)
    print(f"how={how:5s} → {len(m)}행")

print()
print(pd.merge(df_a, df_b, on='gameId', how='outer'))
```
</details>

In [ ]:
# ─── Pong 6: 여기에 코드를 작성하시오 ────────────────────────


---
## 2부 연습문제 (10문항)

In [ ]:
# ─── 연습문제 기본 데이터 ────────────────────────────────────
ex2 = pd.DataFrame({
    'name'  : ['Alice','Bob','Charlie','Dave','Eve'],
    'score' : [78, 92, 65, 88, 70],
    'email' : ['a@gmail.com','b@naver.com','c@gmail.com','d@daum.net','e@naver.com'],
})
score_detail = pd.DataFrame({
    'name'  : ['Alice','Bob','Charlie','Frank'],
    'midterm': [75, 90, 60, 85],
    'final' : [80, 95, 70, 88],
})
print(ex2)
print()
print(score_detail)

### 연습문제 1. map으로 등급화

`score`를 `{90이상:'A', 80이상:'B', 70이상:'C', 그외:'D'}`로 등급화하여 `grade` 열을 만들라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex2['grade'] = ex2['score'].apply(lambda x: 'A' if x>=90 else ('B' if x>=80 else ('C' if x>=70 else 'D')))
print(ex2)
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 2. lambda로 합격/불합격

`score >= 75`이면 `'합격'`, 아니면 `'불합격'`을 `result` 열로 만들라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex2['result'] = ex2['score'].apply(lambda x: '합격' if x>=75 else '불합격')
print(ex2)
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 3. DataFrame.apply로 행 조합

`score`가 85 이상인 사람만 `name`과 `score`를 묶어 `'Alice(78)'` 형태 문자열 시리즈를 만들라. (axis=1 사용)

<details>
<summary>▶ 정답 보기</summary>

```python
ex2.apply(lambda r: f"{r['name']}({r['score']})", axis=1)
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 4. str.lower

`email`을 소문자로 변환한 시리즈를 출력하라 (이미 소문자지만 일반화 연습).

<details>
<summary>▶ 정답 보기</summary>

```python
ex2['email'].str.lower()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 5. str.contains

`email`에 `'gmail'`을 포함한 행만 추출하라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex2[ex2['email'].str.contains('gmail')]
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 6. str.split

`email`을 `@`로 분리하여 **도메인만** `domain` 열로 저장하라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex2['domain'] = ex2['email'].str.split('@').str[1]
print(ex2)
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 7. inner merge

`ex2`와 `score_detail`을 `name`으로 `inner` merge하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.merge(ex2, score_detail, on='name', how='inner')
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 8. left merge

`left` merge 결과의 행 수를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
len(pd.merge(ex2, score_detail, on='name', how='left'))
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 9. outer merge

`outer` merge 결과를 출력하고 결측값이 들어간 칸을 확인하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.merge(ex2, score_detail, on='name', how='outer')
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 10. concat 수직

`ex2`를 자기 자신과 수직으로 이어붙여 행이 2배가 된 결과의 **행 수**를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
len(pd.concat([ex2, ex2], ignore_index=True))
```
</details>

In [ ]:
# 여기에 코드 작성


---
# 3부: 집계와 재구조화 — `groupby` · `crosstab` · `pivot` · `melt`

> **핵심 질문**: *"수만 행의 데이터에서 의미 있는 패턴을 어떻게 한 줄로 요약할 것인가?"*

---

## 3부 학습 맵

```
  ① groupby 심화 — 여러 집계 함수, 다중 키   (Ping 7  / Pong 7)
        예: 성별 × 나이대 평균 콜레스테롤
        ↓
  ② crosstab — 범주형 교차표                   (Ping 8  / Pong 8)
        예: 성별 × 심장질환 여부 빈도표
        ↓
  ③ pivot_table — 엑셀 피벗의 완전판           (Ping 9  / Pong 9)
        예: 성별 × 가슴통증타입 → 평균 나이
        ↓
  ④ stack / unstack / melt — 표 구조 변환     (Ping 10 / Pong 10)
        Wide ↔ Long 전환 (시각화/모델링 전처리)
```

## 3-1. `groupby` 심화 — `agg()`로 여러 함수를 한 번에

### DAY 1 복습 + 확장

DAY 1에서는 `df.groupby('열')['값열'].mean()` 한 줄만 다뤘다. 실무에서는 이보다 더 강력한 **세 가지 패턴**이 필요하다.

### 패턴 1 — `agg()`로 여러 함수를 동시 적용

```python
df.groupby('Sex')['Cholesterol'].agg(['mean','std','min','max','count'])
```

→ 결과는 **데이터프레임**. 각 그룹마다 5개 통계량이 한 줄에 정리된다.

### 패턴 2 — 다중 키(여러 열로 그룹핑)

```python
df.groupby(['Sex', 'ChestPainType'])['Age'].mean()
```

→ 결과는 **멀티인덱스 시리즈**. `.unstack()`으로 표 형태로 바꿀 수 있다.

### 패턴 3 — 열마다 다른 함수 적용 (딕셔너리)

```python
df.groupby('Sex').agg({'Age':'mean', 'Cholesterol':['min','max'], 'HeartDisease':'sum'})
```

> **AI 연결**: 군집(**Clustering**) 결과 분석, 클래스 불균형 확인, 피처 엔지니어링 등 **ML 전 과정**에서 groupby가 핵심 도구다.

### Ping 7 — groupby 심화 (heart.csv)

In [ ]:
# ─── Ping 7: groupby 심화 ────────────────────────────────────
df7 = df_heart.copy()

# [1] 단일 그룹 + 여러 함수
summary1 = df7.groupby('Sex')['Cholesterol'].agg(['mean','std','min','max','count'])
print("▶ 성별 × 콜레스테롤 5대 통계")
print(summary1.round(2))
print()

# [2] 다중 그룹
summary2 = df7.groupby(['Sex','ChestPainType'])['Age'].mean().round(1)
print("▶ 성별 × 가슴통증타입 × 평균 나이 (멀티인덱스 시리즈)")
print(summary2)
print()

# .unstack()으로 표 형태로
summary2_wide = summary2.unstack()
print("▶ 위의 결과를 unstack() → 2차원 표")
print(summary2_wide)
print()

# [3] 열마다 다른 집계
summary3 = df7.groupby('Sex').agg({
    'Age'          : 'mean',
    'Cholesterol'  : ['min','max'],
    'HeartDisease' : 'sum'
}).round(2)
print("▶ 열별로 다른 집계 함수")
print(summary3)
print()

# [4] named aggregation (pandas 0.25+, 권장 문법)
summary4 = df7.groupby('Sex').agg(
    avg_age          = ('Age', 'mean'),
    chol_range       = ('Cholesterol', lambda x: x.max() - x.min()),
    disease_count    = ('HeartDisease', 'sum'),
    patient_count    = ('Age', 'count')
).round(2)
print("▶ named aggregation (결과 열 이름 명시)")
print(summary4)

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(13,4))
summary1['mean'].plot(kind='bar', ax=axes[0], color=['steelblue','crimson'])
axes[0].set_title('성별 평균 콜레스테롤')
axes[0].set_ylabel('mg/dL')
axes[0].tick_params(axis='x', rotation=0)

summary2_wide.plot(kind='bar', ax=axes[1])
axes[1].set_title('성별 × 가슴통증타입 × 평균 나이')
axes[1].set_ylabel('나이(세)')
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

# [AI 연결]
print()
print("[AI 연결] 클래스 불균형 확인, 피처 엔지니어링, 군집 결과 분석에 모두 쓰인다.")

### ✎ 개념 확인 7

**질문**: `groupby(['A','B'])['X'].mean()`의 결과 타입은?

<details>
<summary>▶ 정답 보기</summary>

**멀티인덱스 시리즈**(MultiIndex Series)이다. 인덱스가 `(A값, B값)` 튜플 형태의 2단 계층이고 값이 평균이다. `.unstack()`을 붙이면 한 축이 열로 이동하여 **2차원 데이터프레임**이 된다.
</details>

### Pong 7 — groupby 심화 (LoL 데이터)

**과제**:
1. `df_lol`에서 `blueFirstBlood × blueDragons`로 그룹핑하여 `blueWins`의 **평균(승률)**을 구하라.
2. 그룹별 `blueGoldDiff`의 **min, max, mean, std**를 `agg()`로 한 번에 뽑으라.
3. **named aggregation**으로 `(avg_gold, win_rate, game_count)`를 `blueDragons`별로 뽑으라.

<details>
<summary>▶ 정답 보기</summary>

```python
# [1] 교차 그룹핑 → 승률
win_rate = df_lol.groupby(['blueFirstBlood','blueDragons'])['blueWins'].mean().round(3)
print("▶ 첫킬 × 용 수 × 승률")
print(win_rate.unstack())

# [2] agg — 여러 함수
gold_stats = df_lol.groupby('blueFirstBlood')['blueGoldDiff'].agg(['min','max','mean','std']).round(1)
print()
print("▶ 첫킬 여부 × 골드 차이 통계")
print(gold_stats)

# [3] named aggregation
dragon_stats = df_lol.groupby('blueDragons').agg(
    avg_gold   = ('blueGoldDiff', 'mean'),
    win_rate   = ('blueWins', 'mean'),
    game_count = ('gameId', 'count')
).round(3)
print()
print("▶ 용 수 × 주요 지표")
print(dragon_stats)
```
</details>

In [ ]:
# ─── Pong 7: 여기에 코드를 작성하시오 ────────────────────────


---
## 3-2. `pd.crosstab()` — 범주형 교차표

### 언제 쓰는가?

두 개 이상의 **범주형 열** 사이의 관계를 **빈도표(frequency table)** 로 정리할 때 쓴다. `groupby().size().unstack()`의 축약형이라고 볼 수 있다.

### 기본 사용

```python
pd.crosstab(행_범주, 열_범주)                      # 빈도
pd.crosstab(행_범주, 열_범주, normalize='index')  # 행별 비율
pd.crosstab(행_범주, 열_범주, values=값, aggfunc='mean')  # 평균
pd.crosstab(행_범주, 열_범주, margins=True)       # 합계 행/열 추가
```

### `normalize` 옵션

| 값 | 의미 |
|:---|:---|
| `'index'` | 각 **행**의 합이 1이 되도록 비율 (행 방향 비율) |
| `'columns'` | 각 **열**의 합이 1이 되도록 비율 (열 방향 비율) |
| `'all'` | 전체 합이 1이 되도록 비율 |

> **AI 연결**: 분류 모델 평가의 **혼동 행렬**(confusion matrix)이 바로 이 crosstab 형태이다.

### Ping 8 — crosstab (heart.csv)

In [ ]:
# ─── Ping 8: crosstab ────────────────────────────────────────
df8 = df_heart.copy()

# [1] 기본 빈도표 — 성별 × 심장질환 여부
ct1 = pd.crosstab(df8['Sex'], df8['HeartDisease'])
print("▶ 성별 × 심장질환 여부 (빈도)")
print(ct1)
print()

# [2] 행별 비율 (성별 기준 발병률)
ct2 = pd.crosstab(df8['Sex'], df8['HeartDisease'], normalize='index').round(3)
print("▶ 성별 기준 발병 비율 (행별 정규화)")
print(ct2)
print()

# [3] 합계 포함
ct3 = pd.crosstab(df8['Sex'], df8['ChestPainType'], margins=True, margins_name='합계')
print("▶ 성별 × 가슴통증타입 (합계 포함)")
print(ct3)
print()

# [4] values + aggfunc = 범주별 평균값
ct4 = pd.crosstab(
    df8['Sex'], df8['ChestPainType'],
    values=df8['Cholesterol'], aggfunc='mean'
).round(1)
print("▶ 성별 × 가슴통증타입 × 평균 콜레스테롤")
print(ct4)

# 시각화 — 히트맵
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
im = ax.imshow(ct4.values, cmap='YlOrRd', aspect='auto')
ax.set_xticks(range(len(ct4.columns)))
ax.set_xticklabels(ct4.columns)
ax.set_yticks(range(len(ct4.index)))
ax.set_yticklabels(ct4.index)
for i in range(len(ct4.index)):
    for j in range(len(ct4.columns)):
        ax.text(j, i, f'{ct4.values[i,j]:.0f}', ha='center', va='center')
ax.set_title('성별 × 가슴통증타입 × 평균 콜레스테롤')
plt.colorbar(im)
plt.tight_layout()
plt.show()

# [AI 연결]
print()
print("[AI 연결] sklearn.metrics.confusion_matrix는 crosstab의 특수형태다.")
print("         혼동 행렬: 실제 클래스 × 예측 클래스 = 맞음/틀림 횟수")

### ✎ 개념 확인 8

**질문**: `pd.crosstab(A, B, normalize='index')`에서 각 행의 합은 얼마인가?

<details>
<summary>▶ 정답 보기</summary>

**1.0**이다. `normalize='index'`는 행별로 합이 1이 되도록 비율을 계산한다. 즉, 각 행은 그 그룹 내부의 분포를 나타낸다. 예를 들어 `Sex='M'` 행은 남성 중 `HeartDisease=0`과 `1`의 비율을 보여주고 두 값을 더하면 1.0이 된다.
</details>

### Pong 8 — crosstab (LoL 데이터)

**과제**:
1. `df_lol`에서 `blueFirstBlood` × `blueWins`의 **빈도 crosstab**을 만들라.
2. `normalize='index'`로 **첫킬 성공 시 승률**을 확인하라.
3. `blueDragons × blueWins`의 빈도표에 **합계**(margins=True)를 포함하여 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
# [1] 기본 빈도
ct_a = pd.crosstab(df_lol['blueFirstBlood'], df_lol['blueWins'])
print("▶ 첫킬 × 승패 (빈도)")
print(ct_a)

# [2] 행별 비율 → 승률
ct_b = pd.crosstab(df_lol['blueFirstBlood'], df_lol['blueWins'], normalize='index').round(3)
print()
print("▶ 첫킬 여부별 승률")
print(ct_b)
print(f"→ 첫킬 성공 시 승률: {ct_b.loc[1,1]*100:.1f}%")

# [3] 합계 포함
ct_c = pd.crosstab(df_lol['blueDragons'], df_lol['blueWins'], margins=True)
print()
print("▶ 용 수 × 승패 (합계 포함)")
print(ct_c)
```
</details>

In [ ]:
# ─── Pong 8: 여기에 코드를 작성하시오 ────────────────────────


---
## 3-3. `pivot_table()` — 엑셀 피벗의 완전판

### `crosstab`과 `pivot_table`의 차이

| 특징 | `crosstab` | `pivot_table` |
|:---|:---:|:---:|
| 기본 집계 | 빈도(count) | 평균(mean) |
| 여러 집계 함수 | ✗ | ✓ (`aggfunc=['mean','std']`) |
| 결측값 채우기 | ✗ | ✓ (`fill_value=0`) |
| 여러 값 열 동시에 | ✗ | ✓ (`values=[...]`) |
| 범주형만 | ✓ | 상관 없음 |

쉽게 말해, `pivot_table`이 **더 일반적이고 강력한 버전**이다.

### 4대 핵심 인자

```python
pd.pivot_table(
    df,
    index    = '행에_올_열',       # 필수
    columns  = '열에_올_열',        # 선택
    values   = '집계할_숫자열',     # 선택 (생략 시 나머지 숫자열 전부)
    aggfunc  = 'mean',             # 기본 평균, 리스트로 여러 개 가능
    fill_value = 0,                # 빈 셀 채움
    margins  = True                # 합계 행/열
)
```

### Ping 9 — pivot_table (heart.csv)

In [ ]:
# ─── Ping 9: pivot_table ─────────────────────────────────────
df9 = df_heart.copy()

# 나이 구간 추가
df9['AgeGroup'] = pd.cut(df9['Age'], bins=[0,40,55,100], labels=['~40','40-55','55+'])

# [1] 기본 피벗 — 성별 × 나이대 × 평균 콜레스테롤
pv1 = pd.pivot_table(df9, index='Sex', columns='AgeGroup',
                     values='Cholesterol', aggfunc='mean').round(1)
print("▶ 성별 × 나이대 × 평균 콜레스테롤")
print(pv1)
print()

# [2] 여러 집계 함수 동시
pv2 = pd.pivot_table(df9, index='Sex', values='Cholesterol',
                     aggfunc=['mean','std','min','max','count']).round(2)
print("▶ 성별 × 콜레스테롤 5대 통계")
print(pv2)
print()

# [3] 여러 값 열 동시
pv3 = pd.pivot_table(df9, index='AgeGroup',
                     values=['Cholesterol','RestingBP','MaxHR'],
                     aggfunc='mean').round(1)
print("▶ 나이대별 3개 지표 평균")
print(pv3)
print()

# [4] margins로 합계 추가
pv4 = pd.pivot_table(df9, index='Sex', columns='ChestPainType',
                     values='HeartDisease', aggfunc='mean',
                     margins=True, margins_name='전체').round(3)
print("▶ 성별 × 가슴통증타입 × 심장질환 발병률 (전체 포함)")
print(pv4)
print()

# [5] fill_value로 빈 셀 처리
pv5 = pd.pivot_table(df9, index='Sex', columns='AgeGroup',
                     values='Oldpeak', aggfunc='max',
                     fill_value=0).round(2)
print("▶ 성별 × 나이대 × Oldpeak 최댓값 (빈 셀 0으로 채움)")
print(pv5)

# [AI 연결]
print()
print("[AI 연결] 피벗 결과를 바로 히트맵으로 그리면 범주형 변수 간 상호작용을")
print("         즉시 시각화할 수 있다. EDA에서 가장 빠른 인사이트 발견 도구다.")

### ✎ 개념 확인 9

**질문**: `pivot_table`과 `crosstab`에서 같은 결과를 얻으려면 각각 어떻게 쓰는가?

<details>
<summary>▶ 정답 보기</summary>

성별 × 질환 빈도표를 만들 때:

```python
# crosstab 방식
pd.crosstab(df['Sex'], df['HeartDisease'])

# pivot_table 방식
pd.pivot_table(df, index='Sex', columns='HeartDisease',
               values='Age', aggfunc='count')
```

`crosstab`은 **빈도 전용 단축 함수**이고, `pivot_table`은 범용이다. 단순 빈도라면 `crosstab`이 더 간결하다.
</details>

### Pong 9 — pivot_table (LoL 데이터)

**과제**:
1. `blueDragons` × `blueFirstBlood` × `blueWins`의 **평균 승률** 피벗을 만들라.
2. `blueTowersDestroyed` 기준으로 `blueGoldDiff`와 `blueExperienceDiff`의 **평균**을 동시에 뽑으라.
3. 위 결과에 `margins=True`를 붙여 전체 평균을 포함하라.

<details>
<summary>▶ 정답 보기</summary>

```python
# [1] 교차 피벗 — 평균 승률
pv_a = pd.pivot_table(df_lol, index='blueDragons', columns='blueFirstBlood',
                      values='blueWins', aggfunc='mean').round(3)
print("▶ 용 × 첫킬 × 평균 승률")
print(pv_a)

# [2] 여러 값 열 동시
pv_b = pd.pivot_table(df_lol, index='blueTowersDestroyed',
                      values=['blueGoldDiff','blueExperienceDiff'],
                      aggfunc='mean').round(1)
print()
print("▶ 파괴한 타워 수별 골드/경험치 차이 평균")
print(pv_b)

# [3] 전체 평균 포함
pv_c = pd.pivot_table(df_lol, index='blueTowersDestroyed',
                      values=['blueGoldDiff','blueExperienceDiff'],
                      aggfunc='mean', margins=True).round(1)
print()
print("▶ 전체 평균 포함")
print(pv_c)
```
</details>

In [ ]:
# ─── Pong 9: 여기에 코드를 작성하시오 ────────────────────────


---
## 3-4. `stack` · `unstack` · `melt` — 표 구조 변환 (Wide ↔ Long)

### Wide 형식 vs Long 형식

```
  [WIDE 형식]                        [LONG 형식]
   한 행 = 한 관측, 열이 여러 지표    한 행 = (관측, 지표) 조합 하나
  ─────────────────────              ─────────────────────
  id  chol  bp   hr                  id  variable  value
  1   210   120  150                 1   chol      210
  2   240   130  165                 1   bp        120
  3   190   115  140                 1   hr        150
                                     2   chol      240
                                     2   bp        130
                                     2   hr        165
                                     ...
```

### 언제 Wide? 언제 Long?

| 용도 | 적합한 형식 |
|:---|:---|
| 보기 편한 요약 표 | Wide |
| `seaborn`/`ggplot` 시각화 | **Long** (권장) |
| ML 훈련 데이터 | Wide (한 행 = 한 샘플) |
| 시계열 + 여러 지표 분석 | Long |

### 변환 함수 총정리

| 변환 | 함수 | 방향 |
|:---|:---|:---|
| Wide → Long | `df.melt()` | 열 → 행 |
| Long → Wide | `df.pivot()` / `pivot_table()` | 행 → 열 |
| MultiIndex 시리즈 → 2D | `.unstack()` | 인덱스 → 열 |
| 2D → MultiIndex 시리즈 | `.stack()` | 열 → 인덱스 |

> **AI 연결**: `seaborn`의 대부분 함수는 Long 형식을 원한다. 모델링 직전에는 다시 Wide로 바꾸는 경우가 많다.

### Ping 10 — stack / unstack / melt (heart.csv)

In [ ]:
# ─── Ping 10: stack / unstack / melt ────────────────────────
df10 = df_heart.copy().head(5)[['Age','Cholesterol','RestingBP','MaxHR']].reset_index(drop=True)
df10['patient'] = ['P001','P002','P003','P004','P005']
df10 = df10[['patient','Age','Cholesterol','RestingBP','MaxHR']]
print("▶ 원본 (Wide 형식)")
print(df10)
print()

# [1] melt() — Wide → Long
long_df = df10.melt(id_vars='patient', var_name='측정항목', value_name='값')
print(f"▶ melt() 결과 (Long 형식): {long_df.shape}")
print(long_df.head(10))
print()

# [2] pivot() — Long → Wide (원상 복귀)
wide_again = long_df.pivot(index='patient', columns='측정항목', values='값')
print("▶ pivot()으로 다시 Wide")
print(wide_again)
print()

# [3] stack() — 2차원 → 멀티인덱스 시리즈
stacked = df10.set_index('patient').stack()
print(f"▶ stack() 결과: 타입={type(stacked).__name__}, 길이={len(stacked)}")
print(stacked.head(10))
print()

# [4] unstack() — 멀티인덱스 시리즈 → 2차원
unstacked = stacked.unstack()
print("▶ unstack()으로 복원")
print(unstacked)
print()

# [5] 실전 활용 — 시각화용 Long 변환
import matplotlib.pyplot as plt
long_for_plot = df_heart[['Age','Cholesterol','RestingBP','MaxHR']].melt(var_name='지표', value_name='값')
fig, ax = plt.subplots(figsize=(9,4))
long_for_plot.boxplot(column='값', by='지표', ax=ax)
ax.set_title('4개 지표의 분포 (Long 변환 후 박스플롯)')
plt.suptitle('')
plt.tight_layout()
plt.show()

# [AI 연결]
print()
print("[AI 연결] seaborn.lineplot, seaborn.boxplot은 Long 형식 입력을 선호한다.")
print("         모델 입력 직전에는 pivot()으로 다시 Wide로 돌려 행=샘플, 열=특성으로 만든다.")

### ✎ 개념 확인 10

**질문**: `melt()`에서 `id_vars`와 `value_vars`는 각각 무엇을 지정하는가?

<details>
<summary>▶ 정답 보기</summary>

- `id_vars`: **변환하지 않고 그대로 유지할 식별자 열**. 예: 환자 ID
- `value_vars`: **녹여낼(melt) 대상 열들**. 지정하지 않으면 `id_vars`를 제외한 모든 열.

결과로 각 행은 `(식별자, 변수명, 값)`의 3열 구조가 되며, 원본이 N행 × M열이었다면 N × (M-id_vars 수)행이 된다.
</details>

### Pong 10 — stack / unstack / melt (LoL 데이터)

**과제**:
1. `df_lol.head(5)[['gameId','blueKills','blueDeaths','blueAssists']]`을 `df_l10`로 가져온다.
2. `melt()`로 `gameId`를 id로 남기고 나머지를 Long 형식으로 변환하라.
3. 변환 결과를 `pivot()`으로 다시 Wide로 복원하라.
4. `df_l10.set_index('gameId').stack()`의 길이를 출력하라.

<details>
<summary>▶ 정답 보기</summary>

```python
df_l10 = df_lol.head(5)[['gameId','blueKills','blueDeaths','blueAssists']].copy()
print("▶ 원본 Wide")
print(df_l10)

# melt
long_l = df_l10.melt(id_vars='gameId', var_name='metric', value_name='value')
print()
print(f"▶ melt 결과: {long_l.shape}")
print(long_l)

# pivot으로 복원
wide_l = long_l.pivot(index='gameId', columns='metric', values='value')
print()
print("▶ pivot 복원")
print(wide_l)

# stack
stacked_l = df_l10.set_index('gameId').stack()
print()
print(f"▶ stack 결과 길이: {len(stacked_l)} (5행 × 3지표 = 15)")
```
</details>

In [ ]:
# ─── Pong 10: 여기에 코드를 작성하시오 ────────────────────────


---
## 3부 연습문제 (10문항)

In [ ]:
# ─── 연습문제 기본 데이터 ────────────────────────────────────
ex3 = pd.DataFrame({
    'team'  : ['A','A','A','B','B','B','C','C','C'],
    'player': ['p1','p2','p3','p4','p5','p6','p7','p8','p9'],
    'role'  : ['탱','딜','서폿','탱','딜','딜','탱','서폿','딜'],
    'kills' : [3, 8, 2, 4, 9, 11, 2, 1, 7],
    'deaths': [4, 3, 5, 2, 4, 3, 6, 4, 2],
    'win'   : [1, 1, 1, 0, 0, 0, 1, 1, 1],
})
print(ex3)

### 연습문제 1. 팀별 평균 킬

`team`별로 `kills`의 평균을 구하라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.groupby('team')['kills'].mean()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 2. 팀별 여러 통계

`team`별로 `kills`의 `mean, std, max`를 `agg`로 한 번에 뽑으라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.groupby('team')['kills'].agg(['mean','std','max'])
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 3. 다중 그룹

`team × role`로 그룹핑해 `kills` 평균을 구하라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.groupby(['team','role'])['kills'].mean()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 4. unstack

위 결과를 `.unstack()`으로 2차원 표로 바꾸라 (결측 셀은 그대로 NaN).

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.groupby(['team','role'])['kills'].mean().unstack()
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 5. named aggregation

`team`별로 `avg_kills=('kills','mean'), win_sum=('win','sum')`을 뽑으라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.groupby('team').agg(avg_kills=('kills','mean'), win_sum=('win','sum'))
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 6. crosstab 빈도

`team × role`의 **빈도**를 crosstab으로 뽑으라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.crosstab(ex3['team'], ex3['role'])
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 7. crosstab 비율

위 빈도표를 **행별 비율**로 정규화하라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.crosstab(ex3['team'], ex3['role'], normalize='index')
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 8. pivot_table

`team × role × kills`의 평균을 pivot_table로 만들고 빈 셀은 0으로 채워라.

<details>
<summary>▶ 정답 보기</summary>

```python
pd.pivot_table(ex3, index='team', columns='role', values='kills', aggfunc='mean', fill_value=0)
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 9. melt

`ex3`에서 `id_vars=['team','player']`로 두고 `kills, deaths`만 녹여서 Long 형식으로 변환하라.

<details>
<summary>▶ 정답 보기</summary>

```python
ex3.melt(id_vars=['team','player'], value_vars=['kills','deaths'], var_name='metric', value_name='val')
```
</details>

In [ ]:
# 여기에 코드 작성


### 연습문제 10. stack 길이

`ex3.set_index(['team','player'])[['kills','deaths','win']].stack()`의 길이를 구하라 (답: 27).

<details>
<summary>▶ 정답 보기</summary>

```python
len(ex3.set_index(['team','player'])[['kills','deaths','win']].stack())
```
</details>

In [ ]:
# 여기에 코드 작성


---
# 마무리: DAY 2 전체 정리 & AI 연결 지도

## 오늘 배운 것 — 11가지 기술

| 번호 | 주제 | 핵심 함수 | 쓰임새 |
|:---:|:---|:---|:---|
| 1 | **결측값** | `isna`, `fillna`, `dropna` | 비어있는 셀 처리 |
| 2 | **타입 변환** | `astype`, `to_numeric` | 정수/실수/범주/불리언 |
| 3 | **날짜** | `to_datetime`, `.dt` | 문자열 → 시계열 |
| 4 | **apply/map** | `apply`, `map`, `lambda` | 사용자 정의 변환 |
| 5 | **문자열** | `.str.contains`, `.str.split` | 텍스트 추출/정제 |
| 6 | **결합** | `merge`, `concat` | 여러 파일 합치기 |
| 7 | **groupby 심화** | `agg`, named agg | 여러 집계 동시 |
| 8 | **crosstab** | `pd.crosstab` | 범주형 교차 빈도표 |
| 9 | **pivot_table** | `pd.pivot_table` | 엑셀 피벗의 완전판 |
| 10 | **stack/unstack** | `.stack`, `.unstack` | 계층 인덱스 변환 |
| 11 | **melt** | `df.melt` | Wide → Long |

---

## AI 파이프라인 지도 (업데이트)

```
  ┌─ 원본 CSV ────────────────────────────────┐
  │                                            │
  │     ↓  pd.read_csv()        (DAY 1)        │
  │                                            │
  │  [RAW DataFrame]                           │
  │                                            │
  │     ↓  isna/fillna/dropna   (DAY 2 §1)     │
  │     ↓  astype/to_datetime   (DAY 2 §1)     │
  │                                            │
  │  [CLEAN DataFrame]                         │
  │                                            │
  │     ↓  apply/map (특성 엔지니어링)          │
  │     ↓  str 메서드 (텍스트 처리)             │
  │     ↓  merge (여러 원천 결합) (DAY 2 §2)   │
  │                                            │
  │  [FEATURE TABLE]                           │
  │                                            │
  │     ↓  groupby/pivot (EDA)  (DAY 2 §3)     │
  │     ↓  melt (시각화용)      (DAY 2 §3)     │
  │                                            │
  │  [인사이트 + 훈련 데이터]                  │
  │                                            │
  │     ↓  .values → numpy                     │
  │     ↓  torch.tensor()                      │
  │                                            │
  │  [AI 모델 입력] → 훈련 시작                 │
  │                                            │
  └────────────────────────────────────────────┘
```

---

## 다음 시간 예고

| 주차 | 예정 주제 |
|:---:|:---|
| W05 | 데이터 시각화 (matplotlib, seaborn 심화) |
| W06 | 머신러닝 기초 — scikit-learn, 전처리 파이프라인 |
| W07 | 분류 모델 — 로지스틱 회귀, 결정 트리, 랜덤 포레스트 |

**수고 많았다.** DAY 2에서 다룬 11가지 기술은 앞으로의 모든 ML/DL 프로젝트에서 **반복해서 쓰는 기본기**이다. 이제 여러분은 실무급 데이터 전처리 능력을 갖추었다.